# Model 5 — Body Damage Probability Estimator (Logistic Regression)

## Research Question
Based on what is visible in a car listing — its age, mileage, and seller type — what is the probability that this car has sustained body damage?

## Introduction
This notebook uses **Logistic Regression** to estimate the probability that a car has body damage, based on listing-visible features only.

**Rules for this notebook:**
- Uses **unscaled data** (`proceed_dataset_without_scaling.csv`)
- You **must use `LogisticRegression`** — and output must be probability via `predict_proba()`
- You **must create the `is_damaged` binary column** before training (see Feature Engineering)
- You **must drop all Team 3 damage columns** from features to avoid data leakage (list provided in Feature Engineering)
- You **may choose different features**, add features, and tune hyperparameters — but you **cannot change the general technique category** (must remain Logistic Regression)

## Data Import

Run this cell as-is. It loads all required libraries and the unscaled dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_curve, auc)

df = pd.read_csv('../../data/proceed_dataset_without_scaling.csv')
print(f"Dataset shape: {df.shape}")
df.head()

## Feature Engineering

Run this cell as-is. It creates the binary target column `is_damaged`:
- **1** = car has body damage (`paint_damage_score > 0`)
- **0** = pristine car

It also lists all Team 3 damage columns that **must not** be used as features (data leakage prevention).

In [ ]:
# Create binary target: 1 = has body damage, 0 = pristine
df['is_damaged'] = (df['paint_damage_score'] > 0).astype(int)
print(f"Damaged cars: {df['is_damaged'].sum()} ({df['is_damaged'].mean()*100:.1f}%)")
print(f"Pristine cars: {(df['is_damaged'] == 0).sum()} ({(df['is_damaged'] == 0).mean()*100:.1f}%)")

# List all Team 3 damage columns to EXCLUDE from features (data leakage prevention)
damage_columns_to_drop = [
    'paint_damage_score', 'total_painted_parts', 'total_changed_parts', 'is_fully_original',
    'sağ_arka_çamurluk_durumu', 'sol_arka_çamurluk_durumu',
    'sağ_ön_çamurluk_durumu', 'sol_ön_çamurluk_durumu',
    'sağ_arka_kapı_durumu', 'sol_arka_kapı_durumu',
    'sağ_ön_kapı_durumu', 'sol_ön_kapı_durumu',
    'arka_kaput_durumu', 'motor_kaputu_durumu',
    'ön_tampon_durumu', 'arka_tampon_durumu', 'tavan_durumu'
]
print(f"\nDropping {len([c for c in damage_columns_to_drop if c in df.columns])} damage columns to avoid leakage.")

## Feature Selection

**TODO (Student Task):** Select your features from the dataset.

- You may use the recommended list below or choose your own columns
- Do **NOT** include any column from `damage_columns_to_drop` — this would cause data leakage
- You may add or remove features based on your analysis

In [ ]:
# TODO: Select features. Do NOT include any column from damage_columns_to_drop.
recommended_features = [
    'Yıl', 'Kilometre', 'Kimden_Sahibinden', 'Kimden_Yetkili Bayiden',
    'Ortalama Kasko', 'Vites Tipi', 'Kasa Tipi_SUV',
    'Yakıt Tipi_Dizel', 'Motor Hacmi', 'Yükseklik'
]
features = [f for f in recommended_features if f in df.columns and f not in damage_columns_to_drop]
target = 'is_damaged'

X = df[features].fillna(df[features].median())
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

## Model Training

**TODO (Student Task):** Train the Logistic Regression model.

- You **must** use `LogisticRegression` from `sklearn.linear_model`
- Tune the `C` parameter (inverse of regularization strength): try `0.01`, `0.1`, `1`, `10`
- General structure: **instantiate → fit → predict**

In [ ]:
from sklearn.linear_model import LogisticRegression

# TODO: Tune C (inverse of regularization strength). Try 0.01, 0.1, 1, 10.
model = LogisticRegression(
    C=1.0,          # TODO: tune this
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]  # probability of damage
print("Logistic Regression model trained.")

## Evaluation

### Classification Report

Displays precision, recall, F1-score, and support for each class (No Damage / Damaged).

> ⚠️ This cell uses placeholder data for illustration. Replace with your actual model outputs after training.

In [ ]:
# ⚠️ Replace y_test and y_pred with your actual outputs after training.
print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['No Damage (0)', 'Damaged (1)']))

### Confusion Matrix

Shows the number of true positives, false positives, true negatives, and false negatives.

> ⚠️ This cell uses placeholder data for illustration. Replace with your actual model outputs after training.

In [ ]:
# ⚠️ Replace y_test and y_pred with your actual outputs after training.
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['No Damage', 'Damaged'],
            yticklabels=['No Damage', 'Damaged'],
            linewidths=1, ax=ax)
ax.set_xlabel('Predicted', fontsize=13)
ax.set_ylabel('Actual', fontsize=13)
ax.set_title('Confusion Matrix — Damage Probability (Logistic Regression)', fontsize=13)
plt.tight_layout()
plt.show()

### ROC Curve

The ROC curve plots the True Positive Rate against the False Positive Rate at various classification thresholds. A higher AUC (Area Under the Curve) indicates a better model.

> ⚠️ This cell uses placeholder data for illustration. Replace with your actual model outputs after training.

In [ ]:
# ⚠️ Replace y_test and y_prob with your actual outputs after training.
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--', label='Random Classifier')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('ROC Curve — Body Damage Probability Estimator', fontsize=14)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### Feature Coefficients

Each bar represents a feature's logistic regression coefficient.
- **Red (positive)** → increases predicted damage probability
- **Green (negative)** → decreases predicted damage probability

> ⚠️ This cell uses placeholder data for illustration. Replace with your actual trained model after training.

In [ ]:
# ⚠️ Replace model and features with your actual trained model after training.
coefs = pd.Series(model.coef_[0], index=features).sort_values()
colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in coefs.values]

fig, ax = plt.subplots(figsize=(10, 6))
coefs.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Logistic Regression Coefficient', fontsize=13)
ax.set_title('Feature Coefficients\n(Red = increases damage probability, Green = decreases it)', fontsize=13)
plt.tight_layout()
plt.show()

### Distribution of Predicted Damage Probabilities

This histogram shows how the model's predicted probabilities are distributed for actually-damaged vs. actually-pristine cars. Well-separated distributions indicate a more discriminative model.

> ⚠️ This cell uses placeholder data for illustration. Replace with your actual model outputs after training.

In [ ]:
# ⚠️ Replace y_prob and y_test with your actual outputs after training.
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(y_prob[y_test == 0], bins=40, alpha=0.6, color='#2ecc71', label='No Damage (Actual)')
ax.hist(y_prob[y_test == 1], bins=40, alpha=0.6, color='#e74c3c', label='Damaged (Actual)')
ax.axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='Decision Boundary (0.5)')
ax.set_xlabel('Predicted Damage Probability', fontsize=13)
ax.set_ylabel('Count', fontsize=13)
ax.set_title('Distribution of Predicted Damage Probabilities\n(Well-separated distributions = good model)', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

### Demo Predictions for 4 Example Cars

Runs the trained model on four manually specified example cars to illustrate how listing-visible features translate into a damage probability estimate.

> ⚠️ This cell uses placeholder data for illustration. Replace with your actual model and feature list after training.

In [ ]:
# ⚠️ Replace with your actual model and feature list after training.
demo_cases = [
    {'label': '2011 model, 305,000 km, private seller',
     'values': {'Yıl': 2011, 'Kilometre': 305000, 'Kimden_Sahibinden': 1,
                'Kimden_Yetkili Bayiden': 0, 'Ortalama Kasko': 15000,
                'Vites Tipi': 0, 'Kasa Tipi_SUV': 0, 'Yakıt Tipi_Dizel': 1,
                'Motor Hacmi': 1600, 'Yükseklik': 145}},
    {'label': '2023 model, 8,000 km, authorized dealer',
     'values': {'Yıl': 2023, 'Kilometre': 8000, 'Kimden_Sahibinden': 0,
                'Kimden_Yetkili Bayiden': 1, 'Ortalama Kasko': 55000,
                'Vites Tipi': 1, 'Kasa Tipi_SUV': 1, 'Yakıt Tipi_Dizel': 0,
                'Motor Hacmi': 2000, 'Yükseklik': 168}},
    {'label': '2017 model, 150,000 km, private seller',
     'values': {'Yıl': 2017, 'Kilometre': 150000, 'Kimden_Sahibinden': 1,
                'Kimden_Yetkili Bayiden': 0, 'Ortalama Kasko': 28000,
                'Vites Tipi': 0, 'Kasa Tipi_SUV': 0, 'Yakıt Tipi_Dizel': 1,
                'Motor Hacmi': 1400, 'Yükseklik': 138}},
    {'label': '2021 model, 35,000 km, dealer',
     'values': {'Yıl': 2021, 'Kilometre': 35000, 'Kimden_Sahibinden': 0,
                'Kimden_Yetkili Bayiden': 1, 'Ortalama Kasko': 40000,
                'Vites Tipi': 1, 'Kasa Tipi_SUV': 0, 'Yakıt Tipi_Dizel': 0,
                'Motor Hacmi': 1800, 'Yükseklik': 150}},
]

print("=== Damage Probability Demo ===\n")
for case in demo_cases:
    row = pd.DataFrame([{f: case['values'].get(f, 0) for f in features}])
    prob = model.predict_proba(row)[0][1]
    print(f"{case['label']} → Damage probability: {prob*100:.0f}%")

## ⚠️ If Your Model Underperforms

If your model produces poor or surprising results (e.g., very low accuracy, unexpected associations, or trivial rules), **do not discard your results**.

- Keep all outputs as-is
- In your presentation, document exactly what you observe
- Write a short hypothesis: Why might the model have failed? (e.g., 'The dataset may not contain enough transactions with rare feature combinations to produce reliable high-confidence rules')